<a href="https://colab.research.google.com/github/suphakornwong/Chapter_4/blob/main/Object_Detection_Train_Method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Things to Prepare Before Coding
1. Yolo Pretraind model downloaded from https://docs.ultralytics.com/models/yolo11
2. API key form Roboflow platform https://roboflow.com/

Code Cell-A

In [1]:
!pip install ultralytics --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


Code Cell-B

In [4]:
import cv2
import torch
import torchvision

print(cv2.__version__, torch.__version__, torchvision.__version__, torch.cuda.is_available())

4.14.0 2.14.1+cu130 0.29.1+cu130 True


Code Cell-C

In [5]:
import os
HOME = os.getcwd()
print(HOME)
%mkdir -p {HOME}/datasets
%cd {HOME}/datasets


/content
/content/datasets


Code Cell-D   # Upload from your computer

In [ ]:
!unzip /content/PestDataBase.zip

Code Cell-E    ## This cell is API key from Roboflow platform

In [6]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="1ngtZRVkSWTfOiBcnMzY")
project = rf.workspace("birdagriculture").project("pigeon_main")
version = project.version(6)
dataset = version.download("yolo26")      # train 24 Sep 26


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 87.8 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [roboflow]
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pigeon_main-6 in yolo26:: 100%|██████████| 33675/33675 [00:05<00:00, 6056.61it/s]


Code Cell-F   # Save your result to google drive

In [7]:
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/Detection'
os.makedirs(folder_path, exist_ok=True)
print(f"Folder created at: {folder_path}")
save_dir = '/content/drive/MyDrive/Detection'


Mounted at /content/drive
Folder created at: /content/drive/MyDrive/Detection


Code Cell-G

In [8]:
import shutil
import random
from tqdm import tqdm

def split_dataset(dataset_dir, output_dir, train_ratio=0.75, val_ratio=0.15, test_ratio=0.1, seed=42):
   assert train_ratio + val_ratio + test_ratio == 1.0, "Ratios must sum to 1"

   random.seed(seed)
   classes = os.listdir(dataset_dir)

   for class_name in classes:
       class_dir = os.path.join(dataset_dir, class_name)
       if not os.path.isdir(class_dir):
           continue

       images = os.listdir(class_dir)
       random.shuffle(images)

       total = len(images)
       train_count = int(train_ratio * total)
       val_count = int(val_ratio * total)

       train_files = images[:train_count]
       val_files = images[train_count:train_count + val_count]
       test_files = images[train_count + val_count:]

       # Copy files
       for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
           split_class_dir = os.path.join(output_dir, split, class_name) # Removed the extra 'data' subdirectory
           os.makedirs(split_class_dir, exist_ok=True)

           for file in tqdm(split_files, desc=f"Copying {split}/{class_name}"):
               src = os.path.join(class_dir, file)
               dst = os.path.join(split_class_dir, file)
               if os.path.isfile(src):  # Add this check
                   shutil.copy2(src, dst)


Code Cell-(I-D)

In [14]:
from ultralytics import YOLO

model = YOLO('/content/yolov8n.pt')      ## Right-click at Pretraind model (for example, yolo8n.pt) in /content, Select "copy path"
model.train(
 data='/content/datasets/Pigeon_main-6/data.yaml',    ## find "data.yaml" in datasets folder, Right-click and Select "copy path"
 task='detect',
 epochs=100,
 imgsz=640,
 device=0
)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/Pigeon_main-6/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opset=Non

KeyboardInterrupt: 